In [1]:
import pandas as pd

In [2]:
from pathlib import Path

In [3]:
raw_dir = Path("../data/raw")

breast = pd.read_csv(
    raw_dir / "hlth_ehis_pa7e__custom_22838814_linear_2_0.csv.gz"
)

cervical = pd.read_csv(
    raw_dir / "hlth_ehis_pa8e__custom_22840528_linear_2_0.csv.gz"
)

breast.shape, cervical.shape

((9996, 23), (10920, 23))

In [4]:
columns_to_keep = [
    "duration",
    "Duration",
    "isced11",
    "International Standard Classification of Education (ISCED 2011)",
    "age",
    "Age class",
    "geo",
    "Geopolitical entity (reporting)",
    "TIME_PERIOD",
    "OBS_VALUE",
    "OBS_FLAG",
    "Observation status (Flag) V2 structure",
]

breast_clean = breast[columns_to_keep].copy()
cervical_clean = cervical[columns_to_keep].copy()

breast_clean.shape, cervical_clean.shape

((9996, 12), (10920, 12))

In [12]:
breast_clean = breast_clean.rename(columns={
    "Duration": "duration_label",
    "International Standard Classification of Education (ISCED 2011)":"education_label",
    "Age class":"age_label",
    "Geopolitical entity (reporting)":"country_name",
    "TIME_PERIOD":"year",
    "OBS_VALUE":"screening_rate",
    "OBS_FLAG":"quality_flag",
    "Observation status (Flag) V2 structure":"quality_flag_label",
})

In [13]:
breast_clean.columns


Index(['duration', 'duration_label', 'isced11', 'education_label', 'age',
       'age_label', 'geo', 'country_name', 'year', 'screening_rate',
       'quality_flag', 'quality_flag_label'],
      dtype='str')

In [14]:
cervical_clean = cervical_clean.rename(columns={
    "Duration": "duration_label",
    "International Standard Classification of Education (ISCED 2011)":"education_label",
    "Age class":"age_label",
    "Geopolitical entity (reporting)":"country_name",
    "TIME_PERIOD":"year",
    "OBS_VALUE":"screening_rate",
    "OBS_FLAG":"quality_flag",
    "Observation status (Flag) V2 structure":"quality_flag_label",
})

In [15]:
cervical_clean.columns

Index(['duration', 'duration_label', 'isced11', 'education_label', 'age',
       'age_label', 'geo', 'country_name', 'year', 'screening_rate',
       'quality_flag', 'quality_flag_label'],
      dtype='str')

In [16]:
breast_clean["screening_type"] = "breast_xray"
cervical_clean["screening_type"] = "cervical_smear"

In [18]:
screening_clean = pd.concat(
    [breast_clean, cervical_clean],
    ignore_index=True
)

screening_clean["screening_type"].value_counts()

screening_type
cervical_smear    10920
breast_xray        9996
Name: count, dtype: int64

In [20]:
screening_clean.groupby("screening_type")["screening_rate"].apply(
    lambda values: values.isna().sum()
)

screening_type
breast_xray       145
cervical_smear    172
Name: screening_rate, dtype: int64

In [21]:
screening_clean = screening_clean.dropna(
    subset=["screening_rate"]
).copy()

screening_clean.shape

(20599, 13)

In [22]:
##скрининг уникального ключа:
key_columns = [
    "screening_type",
    "geo",
    "year",
    "age",
    "isced11",
    "duration",
]

screening_clean.duplicated(subset=key_columns).sum()

np.int64(0)

In [23]:
#проверка типов данных:
screening_clean.dtypes

duration                  str
duration_label            str
isced11                   str
education_label           str
age                       str
age_label                 str
geo                       str
country_name              str
year                    int64
screening_rate        float64
quality_flag              str
quality_flag_label        str
screening_type            str
dtype: object

In [24]:
screening_clean = screening_clean.sort_values(
    by=[
        "screening_type",
        "geo",
        "year",
        "age",
        "isced11",
        "duration",
    ]
).reset_index(drop=True)

processed_path = Path("../data/processed/screening_clean.csv")

screening_clean.to_csv(
    processed_path,
    index=False
)

processed_path

WindowsPath('../data/processed/screening_clean.csv')

выбраны 12 полезных исходных столбцов;
названия приведены к понятному формату;
добавлен screening_type;
две таблицы объединены;
удалены 317 строк без screening_rate, но исходные файлы сохранены;
флаги качества сохранены;
дубликатов по аналитическому ключу нет;
результат сохранён в data/processed/screening_clean.csv